# 📗 LangChain 에이전트와 도구 — 구조화된 출력·도구·MCP

> 엔코아 AI캠퍼스 · 데이터 분석 & AI 머신러닝 캠프

지난 시간까지 우리는 모델을 부르고, 프롬프트를 템플릿으로 만들고, 파이프 `|` 로 체인을 이었습니다. 그렇게 받은 답은 언제나 **사람이 읽는 문장**이었습니다. 문장은 사람에게는 좋지만, **프로그램에게는 다루기 어려운 덩어리**입니다.

이번 시간에는 두 가지를 배웁니다. 첫째, 모델의 답을 **정해진 칸에 담긴 데이터**로 받는 **구조화된 출력** — 지난 단원에서 손으로 하던 정형화를 **LangChain 부품 한 줄**로 바꿉니다. 둘째, 모델이 **모르는 것**을 알아 오게 하는 **도구(tool)** — 그리고 그 도구를 언제 쓸지 모델이 스스로 정하는 **에이전트(agent)** 입니다. 마지막으로 남이 만들어 둔 도구 서버에 붙는 표준 규격 **MCP** 까지 봅니다.

## ⏪ 복습 — 지난 시간: 모델·프롬프트·체인·Runnable

- **모델**: `ChatOpenAI(model='gpt-4o-mini', temperature=0)` 로 만들고 `.invoke('질문')` 으로 답을 받았습니다. 답은 `AIMessage` 부품이고, 순수 텍스트는 `.text` 로 꺼냈습니다.
- **프롬프트 템플릿**: `ChatPromptTemplate.from_template('...{변수}...')` 로 값만 갈아 끼우는 프롬프트를 만들었습니다.
- **출력 파서**: `StrOutputParser()` 로 응답에서 **문자열만** 뽑았습니다.
- **체인(LCEL)**: `프롬프트 | 모델 | 파서` 를 파이프로 이어 `invoke`·`batch` 로 한 번에 돌렸습니다.
- **Runnable**: 모든 부품이 같은 규약(`invoke`·`batch`·`stream`)을 따르기에 서로 이어 붙일 수 있었습니다.

여기까지의 결과물은 전부 **자유 문장**이었습니다. 오늘은 그 답을 **칸에 담고**, 모델이 **혼자서는 알 수 없는 것**까지 답하게 만듭니다.

### 📚 공식 문서 — 오늘 배우는 것들

막히거나 더 알고 싶을 때 **가장 먼저 볼 곳**입니다. LangChain 은 버전이 빨리 올라가니 블로그·오래된 예제보다 **공식 문서**를 먼저 확인하는 습관을 들이세요(이 교재는 **LangChain 1.x** 기준).

| 오늘 다루는 것 | 공식 문서 |
|---|---|
| **구조화된 출력** — `model.with_structured_output(스키마)` | [Models](https://docs.langchain.com/oss/python/langchain/models) 의 Structured output 절 |
| 스키마 필드 선언 `Literal`·`Optional`·`description` | [pydantic Field](https://docs.pydantic.dev/latest/concepts/fields/) |
| **도구** — `@tool` 정의와 명세 | [Tools](https://docs.langchain.com/oss/python/langchain/tools) · [`@tool` 레퍼런스](https://reference.langchain.com/python/langchain-core/tools/) |
| **에이전트** — `create_agent` | [Agents](https://docs.langchain.com/oss/python/langchain/agents) |
| **메시지 궤적** — Human/AI/Tool 메시지 | [Messages](https://docs.langchain.com/oss/python/langchain/messages) |
| **에이전트의 구조화 응답** — `create_agent(response_format=...)`·`ProviderStrategy` | [Structured output (agents)](https://docs.langchain.com/oss/python/langchain/structured-output) |
| **MCP** — 남이 만든 도구 서버에 붙기 | [MCP](https://docs.langchain.com/oss/python/langchain/mcp) · [규격 자체](https://modelcontextprotocol.io/docs/getting-started/intro) |
| OpenAI SDK 를 직접 쓰던 단원과의 연결(같은 개념의 OpenAI 판) | [OpenAI Structured Outputs](https://platform.openai.com/docs/guides/structured-outputs) |

> **두 사이트의 역할이 다릅니다.** `docs.langchain.com` 은 **개념과 사용법**을 설명하는 안내서이고, `reference.langchain.com` 은 **클래스·인자 목록**을 그대로 보여 주는 사전입니다. "이게 왜 필요한가"는 앞쪽, "이 함수에 어떤 인자가 있나"는 뒤쪽에서 찾으면 빠릅니다.

**오늘의 목표**

- [ ] **구조화된 출력** — `with_structured_output` 으로 모델의 답을 **정해진 칸**에 담아 받는다.
- [ ] **스키마 설계 복습** — `Literal`·`Optional`·`description` 을 언제 왜 쓰는지 되짚는다.
- [ ] **부품으로 재사용** — 분류기를 한 번 만들어 여러 문의에 돌리고, 모델 회사를 바꿔도 스키마·호출이 그대로임을 안다.
- [ ] **도구 만들기** — `@tool` 로 함수를 도구로 바꾸고, docstring·타입힌트가 **명세**가 됨을 확인한다.
- [ ] **에이전트** — `create_agent(model, 도구들)` 로 만들고 **메시지 궤적**을 읽는다.
- [ ] **에이전트의 구조화 응답** — `response_format` 으로 도구를 쓴 결과까지 스키마에 담아 받는다.
- [ ] **쓸모 있는 도구의 조건** — 인자 다듬기·실패 처리·권한 범위를 적용한다.
- [ ] **실무형 도구** — 사내 데이터 조회·규칙 계산·외부 REST API 를 도구로 만든다.
- [ ] **MCP** — 이미 만들어져 있는 도구 서버에 붙어 그 도구를 에이전트에 넘긴다.

아래 준비 셀들을 먼저 실행하세요(지난 시간과 같은 `.env` 의 `OPENAI_API_KEY` 를 씁니다). 오늘 오가는 요청은 짧은 문장 수십 건이라 비용은 아주 적습니다.

In [ ]:
# [제공 코드] OpenAI 키 준비 — 이 셀은 실행만 하세요.
# 14~16일차와 같은 방식입니다: .env 파일에 넣어 둔 OPENAI_API_KEY 를 읽어 옵니다.
import os

from dotenv import load_dotenv

load_dotenv(".env")       # 같은 폴더의 .env
load_dotenv("../.env")    # 정답 폴더에서 실행하는 경우

# 키를 먼저 확인합니다 — 모델을 만든 뒤에 검사하면 인증 오류가 먼저 나서 이 안내가 묻힙니다.
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(
        "이 노트북은 실제 OpenAI 호출이 필요합니다 — OPENAI_API_KEY 를 찾지 못했습니다.\n"
        "  1) 일차 폴더에서  cp .env.example .env\n"
        "  2) .env 를 열어 본인 키를 채우세요\n"
        "  3) 커널을 재시작한 뒤 이 셀부터 다시 실행하세요")

print("OpenAI 키 확인 완료 — 이제 LangChain 으로 모델을 만들 수 있습니다.")

In [ ]:
# [제공 코드] 이번 시간 공통 부품 — 지난 시간에 배운 모델
from langchain_openai import ChatOpenAI

# temperature=0 : 같은 질문에 되도록 일정한 답을 받는 설정(수업·채점용).
model = ChatOpenAI(model='gpt-4o-mini', temperature=0)

---
# 1. 구조화된 출력 — 답을 시스템이 쓸 데이터로 받기

> **정형화와 스키마 설계는 이미 익힌 내용입니다.** **OpenAI SDK 를 직접 쓰던 단원**에서 문의를 갈래·긴급도로 나누고, 제각각으로 적힌 설문 응답을 정해진 값으로 표준화하고, 그렇게 만든 표를 세어 봤지요. 이 절은 그것을 **되짚은 다음**, 오늘 달라지는 **한 줄**에 집중합니다.

## 왜 필요할까요? — 되짚기
사내 헬프데스크에 이런 문의가 하루 300건 들어온다고 해 봅시다.

```
"사무실 노트북(IT-2043)이 어제부터 전원이 안 들어옵니다. 오늘 오후에 발표가 있어서 급합니다."
```

이 상태로는 **"하드웨어 문의가 몇 건인가", "급한 건 몇 건인가"** 를 셀 수 없습니다. 세려면 각 문장을 **같은 칸**을 가진 행으로 바꿔야 했습니다.

| 원문 | 갈래 | 긴급도 | 관리번호 |
|---|---|---|---|
| 사무실 노트북(IT-2043)이 어제부터… | `하드웨어` | `긴급` | IT-2043 |

이 변환이 **정형화(structuring)** 였습니다. 정형화가 끝나면 그 뒤는 이미 아는 pandas 세계였지요 — `value_counts()`·필터·저장.

<img src="images/구조화출력_문장_vs_스키마.png" width="820">

*왼쪽처럼 크기도 모양도 제각각인 문장 더미로는 셀 수가 없습니다. 오른쪽처럼 **같은 칸**을 갖게 만드는 것이 정형화입니다 — 오늘 달라지는 것은 이 일을 **무엇으로 하는가** 뿐입니다.*

## 앞서 배운 OpenAI SDK 와의 연결 — 같은 일을, 부품으로

| | 그때 (OpenAI SDK 직접 호출) | 오늘 (LangChain) |
|---|---|---|
| 스키마 선언 | `class Ticket(BaseModel): ...` | **똑같다** |
| 호출 | `client.chat.completions.parse(model=…, response_format=Ticket)` | `model.with_structured_output(Ticket).invoke(문의)` |
| 결과 꺼내기 | `resp.choices[0].message.parsed` | **바로 그 객체**가 돌아온다 |
| 모델 회사를 바꾸면 | 호출 코드를 다시 씀 | 모델 부품 만드는 줄만 교체 |

> **마지막 줄이 이 절을 하는 이유입니다.** 스키마도 그대로, `invoke` 도 그대로 두고 `ChatOpenAI(...)` 만 다른 회사의 모델 부품으로 바꾸면 정형화가 그대로 돕니다. SDK 를 직접 부를 때는 회사마다 함수 이름도, 결과를 꺼내는 경로(`resp.choices[0].message.parsed`)도 달라 **호출 코드를 다시 써야** 했습니다.

## 문법 — `with_structured_output`
`model.with_structured_output(스키마)` 는 **모델에 스키마를 씌운 새 부품**을 돌려줍니다. 그 부품을 `invoke` 하면 답이 문장이 아니라 **그 스키마의 객체**로 돌아옵니다. 이 절에서 새로 외울 것은 이 한 줄뿐입니다.

```python
classifier = model.with_structured_output(스키마)   # 스키마를 씌운 '새 부품'
result = classifier.invoke('문장')                  # 문장이 아니라 스키마 객체가 온다
```


## 복습 — 스키마 설계 규칙

넘기는 스키마는 **pydantic 그대로**라 새로 익힐 것이 없습니다. 지난 단원에서 "이렇게 설계했었죠" 하던 세 가지 원칙과, 그것을 **필드에 어떻게 적는지**를 한자리에 모았습니다. **바로 아래 `Ticket`** 이 이 세 가지를 그대로 담고 있습니다.

| 원칙 | 이렇게 적는다 | 무엇을 정하는가 |
|---|---|---|
| **셀 값은 `Literal` 로 못박는다** | `category: Literal['하드웨어','계정',…]` | 정해진 목록 중 **하나만** 허용(벗어난 값은 거부) |
| **없을 수 있는 값은 `Optional`** | `asset_tag: Optional[str] = Field(default=None)` | 없을 수 있는 값 — 없으면 **`None`** |
| **필드마다 `description`** | `Field(description='IT-#### 형식, 없으면 null')` | 그 필드가 무엇인지 알려 주는 **지침**(모델이 읽는다) |

> **`Literal`** — 지난 단원에서 분류 이름을 `str` 로 열어 두었더니 **측면 이름이 29가지**로 흩어지고 (`톤업효과`/`톤업 효과`) 그중 24개가 딱 한 번씩만 나왔던 그 사례를 떠올리세요. 값이 고정돼야 셀 수 있습니다.
>
> **`Optional`** — "없으면 null" 지침이 빠지면 모델은 빈칸을 싫어해 **그럴듯한 관리번호를 지어냅니다.** 관리번호가 있는 문의와 없는 문의를 나란히 넣어 확인했던 그 함정이고, **정형화에서 가장 위험한 실수**였습니다.
>
> **`description`** — 주석이 아니라 **모델에게 전달되는 문서**입니다. 스키마를 잘 적는 것이 프롬프트를 잘 쓰는 것과 같은 일이 됩니다.

정리하면, **스키마를 적는 방법은 지난 단원과 글자 하나 다르지 않습니다.** 오늘 달라지는 것은 **이 스키마를 어디에 넘기는가** 뿐입니다.

In [ ]:
# 헬프데스크 문의 한 건을 담을 스키마 — 지난 단원에서 쓰던 것과 똑같은 pydantic 클래스입니다.
from typing import Literal, Optional

from pydantic import BaseModel, Field


class Ticket(BaseModel):
    """헬프데스크 문의 한 건을 분류한 결과."""

    # Literal(복습) : 값을 이 목록 안으로 못박는다 -> 나중에 세고 묶을 수 있다.
    category: Literal['하드웨어', '소프트웨어', '계정', '네트워크', '기타'] = Field(
        description='문의가 어떤 갈래인지')
    urgency: Literal['긴급', '보통', '낮음'] = Field(
        description='긴급: 업무가 지금 멈춤 / 보통: 불편하지만 진행 가능 / 낮음: 단순 질문')
    # Optional(복습) : 없을 수 있는 값. 없으면 None 이 되어야 한다(억지로 채우면 지어낸다).
    asset_tag: Optional[str] = Field(
        default=None, description='IT-#### 형태의 자산 관리번호. 문의에 없으면 반드시 null')
    summary: str = Field(description='문의 내용을 한 문장으로 요약')


# 여기까지 LangChain 은 한 번도 나오지 않았다 - 순수 pydantic 이라 지난 단원 코드를 그대로 옮겨 온 것이다.
print('스키마 필드:', list(Ticket.model_fields))

이제 이 스키마를 모델에 **씌워서** 문의 한 건을 분류합니다. 프롬프트에 "JSON 으로 답해줘" 같은 말을 한 마디도 쓰지 않는다는 점은 지난 단원과 같습니다 — **스키마가 곧 지시**입니다. 오늘 눈여겨볼 것은 결과의 **종류**입니다: 파싱해야 할 문자열도, 딕셔너리도 아니고 **`Ticket` 객체**가 그대로 돌아옵니다.

In [ ]:
# 헬프데스크에 들어온 문의 한 건입니다.
INQ_URGENT = '사무실 노트북(IT-2043)이 어제부터 전원이 안 들어옵니다. 오늘 오후에 발표가 있어서 급합니다.'

# with_structured_output : 모델에 스키마를 씌운 '새 부품'을 만든다. 답이 문장 대신 Ticket 객체로 온다.
ticket = model.with_structured_output(Ticket).invoke(INQ_URGENT)

# 확인 포인트: 돌아온 것이 문자열이 아니라 Ticket 이라는 것.
print('결과 종류:', type(ticket).__name__)
print(ticket)
print('갈래만 꺼내기:', ticket.category)   # 점(.)으로 칸 하나를 바로 꺼낸다

## 부품이니까 만들어 두고 계속 씁니다

정형화의 결승선은 결국 **세고 묶는 것**이고, 거기까지는 지난 단원에서 이미 달려 봤습니다 — `model_dump()` 로 객체를 딕셔너리로 바꾸면 딕셔너리 리스트는 곧 `DataFrame` 이고, 그 뒤는 `value_counts()` 하는 판다스 세계였지요. 도입부에서 던진 **"급한 건 몇 건인가"** 에 답할 수 있게 되는 지점도 거기였습니다.

오늘 새로 챙길 감각은 다른 데 있습니다. `with_structured_output` 이 돌려준 것은 **부품**이라는 점입니다. 한 번 만들어 변수에 담아 두면 문의가 몇 건이든 **그 부품에 넣기만** 하면 됩니다.

그리고 부품이라는 말에는 값이 하나 더 붙어 있습니다 — **지난 시간에 배운 `invoke`·`batch`·`stream` 규약이 이 부품에도 그대로 적용됩니다.** 여러 건을 처리할 때 `for` 문으로 하나씩 `invoke` 할 이유가 없습니다. `batch` 에 리스트째 넘기면 **동시에** 보내고 결과는 **넣은 순서 그대로** 돌아옵니다.

In [ ]:
# 분류기를 '한 번' 만들어 변수에 담아 둡니다 — 문의마다 다시 씌울 필요가 없습니다.
classifier = model.with_structured_output(Ticket)

inquiries = [
    '사내 메신저 알림이 안 오는데 어디서 켜나요?',
    '사내 포털 비밀번호를 세 번 틀려서 계정이 잠겼습니다. 풀어 주세요.',
    '재택인데 VPN 이 연결됐다 끊겼다 합니다. 지금 업무가 멈춰 있습니다.',
]

# 부품이니 batch 가 그대로 먹는다 — 세 건을 동시에 보내고, 결과는 넣은 순서대로 온다.
tickets = classifier.batch(inquiries)

# 결과가 객체라 파싱 없이 점(.)으로 칸을 바로 꺼내 쓴다.
for t in tickets:
    print(f'{t.category} / {t.urgency} / {t.summary}')

### 🖐️ 함께 따라하기 — 회의실 예약 문의를 부품 하나로

이번에는 **다른 도메인**으로 연습합니다. 총무팀에 들어오는 **회의실 예약 문의**를 스키마에 담아 보세요. 스키마를 적는 방법은 지난 단원과 같으니, 이 따라하기의 초점은 **부품을 하나 만들어 두 문의에 재사용하는 것**입니다.

1. `RoomRequest` 스키마를 만듭니다.
   - `room_type`: `Literal['소회의실', '대회의실', '세미나실']`
   - `head_count`: `int` — 참석 인원
   - `preferred_date`: `Optional[str]` — 'YYYY-MM-DD' 형태, 문의에 없으면 `None`
   - 모든 필드에 `Field(description=...)` 을 답니다.
2. `room_parser = model.with_structured_output(RoomRequest)` 로 부품을 **한 번만** 만듭니다.
3. 아래 두 문의를 리스트에 담아 **`batch` 한 번**으로 처리하고, 결과의 종류(`type(...).__name__`)와 `.room_type`·`.head_count` 를 출력합니다.
   - `'다음 주 화요일(2026-08-18)에 12명이 들어갈 회의실이 필요합니다.'`
   - `'세미나실 하루 빌리려면 어떻게 신청하나요? 인원은 40명입니다.'`

**확인 기준**: 부품은 한 번만 만들고 `batch` 도 한 번만 불렀는데 결과가 **두 개**, 그것도 **넣은 순서 그대로** 나온다. 두 결과 모두 `RoomRequest` 객체라 문자열 파싱 없이 **점(.)으로 값이 바로** 꺼내진다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) RoomRequest(BaseModel) 스키마를 만든다 (room_type Literal / head_count int /
#    preferred_date Optional[str], 각 필드에 Field(description=...))
# 2) room_parser = model.with_structured_output(RoomRequest) 로 부품을 한 번만 만든다
# 3) 두 문의를 리스트에 담아 room_parser.batch(...) 한 번으로 처리한다
# 4) 결과를 돌며 type(...).__name__ 과 .room_type / .head_count 를 출력한다

### ✅ 바로 확인 퀴즈

**1.** (복습) 분류 값에 `Literal` 을 쓰는 이유는 무엇인가요?

<details><summary>정답 보기</summary>

**집계할 수 있게 값을 고정**하기 위해서입니다. 자유 문자열로 두면 같은 뜻이 여러 이름으로 흩어져 (`톤업효과`/`톤업 효과`처럼) 세거나 묶을 수 없게 됩니다.

</details>

**2.** (복습) `Field(description=...)` 은 사람을 위한 주석인가요?

<details><summary>정답 보기</summary>

아닙니다. **모델에게 전달되는 작성 지침**입니다. 스키마를 잘 적는 것이 프롬프트를 잘 쓰는 것과 같은 일이 됩니다.

</details>

**3.** OpenAI SDK 를 직접 쓰던 단원의 `client.chat.completions.parse(response_format=Ticket)` 에 대응하는 오늘의 코드는?

<details><summary>정답 보기</summary>

`model.with_structured_output(Ticket).invoke(문의)` 입니다. **스키마를 적는 방법(pydantic)은 완전히 같고**, 호출하는 줄만 바뀝니다. 돌아오는 것은 문장이 아니라 **`Ticket` 객체**이고, 그 부품은 변수에 담아 여러 문의에 재사용합니다.

</details>

---
# 2. 도구 만들기 — `@tool`

## 왜 필요할까요?
구조화된 출력은 강력하지만 한계가 분명합니다. **모델이 이미 아는 것**을 정해진 모양으로 받아 낼 뿐입니다. 그런데 실무 질문의 상당수는 모델이 **알 수 없는 것**을 묻습니다.

- "무선 마우스 창고에 몇 개 남았어?" — 우리 창고 데이터는 모델이 본 적이 없습니다.
- "오늘 1달러는 몇 원이야?" — 오늘의 환율은 학습 이후의 일입니다.
- "IT-2043 담당자가 누구야?" — 사내 자산 대장은 회사 안에만 있습니다.

이럴 때 모델에게 **도구**를 쥐어 줍니다. 도구는 거창한 것이 아니라 **그냥 파이썬 함수**입니다.

## 문법 — docstring 이 명세, 타입힌트가 인자 타입
함수 위에 **`@tool`** 을 붙이면 LangChain 이 그 함수를 모델이 쓸 수 있는 도구로 바꿔 줍니다. 이때 **함수의 어디를 읽는지**가 중요합니다.

| 함수의 이 부분이 | 도구의 이것이 된다 | 모델에게 무슨 뜻인가 |
|---|---|---|
| 함수 이름 | `.name` | 이 도구를 부를 때 쓰는 이름 |
| **docstring** | `.description` | **"언제 이 도구를 쓰는가"** — 선택의 근거 |
| **타입힌트** (`item_name: str`) | `.args` 의 타입 | 어떤 인자를 어떤 형으로 넘길지 |

> **모델은 우리 코드를 못 봅니다.** 함수 본문이 아무리 훌륭해도 모델이 읽는 것은 **이름과 설명뿐**입니다. docstring 은 주석이 아니라 **명세서**입니다.

<img src="images/도구_명세_흐름.png" width="820">

*파이썬 함수의 이름·docstring·타입힌트가 JSON 명세로 바뀌어 모델에게 전달됩니다.*

In [ ]:
# 도구는 그냥 파이썬 함수입니다 — 함수 위에 @tool 을 붙이는 것이 전부입니다.
from langchain_core.tools import tool

# 사내 비품 창고(실무에서는 이 자리가 데이터베이스나 사내 시스템입니다)
_STOCK = {'무선 마우스': 40, '기계식 키보드': 15, '복사용지': 200, '보안 USB': 10}
_PRICE = {'무선 마우스': 18000, '기계식 키보드': 65000, '복사용지': 4500, '보안 USB': 32000}


@tool
def supply_stock(item_name: str) -> int:
    """비품 이름(item_name)을 받아 현재 창고 재고 수량을 돌려준다."""
    return _STOCK.get(item_name.strip(), 0)


@tool
def restock_cost(item_name: str, count: int) -> int:
    """비품 이름(item_name)과 재입고 수량(count)을 받아 총 재입고 비용(원)을 돌려준다."""
    return _PRICE.get(item_name.strip(), 0) * count


print('도구 준비:', supply_stock.name, '/', restock_cost.name)

도구가 어떤 **명세**를 갖게 됐는지 들여다봅니다. 셋 다 우리가 따로 등록한 것이 아니라 **함수에서 자동으로 뽑힌 값**입니다.

In [ ]:
# 이름·설명·인자 — 함수의 어느 부분에서 왔는지 짚어 보며 읽으세요.
print('name       :', supply_stock.name)          # 함수 이름에서
print('description:', supply_stock.description)   # docstring 에서
print('args       :', supply_stock.args)          # 타입힌트에서

위 세 가지는 사람이 보기 좋게 나눠 찍은 것이고, 모델에게 실제로 넘어가는 것은 **한 덩어리의 JSON 명세**입니다. OpenAI SDK 를 직접 쓰던 단원에서 함수 호출을 손으로 짤 때 `{"name": ..., "parameters": {...}}` 딕셔너리를 직접 적었던 것을 기억하세요 — `@tool` 은 **함수를 읽어 그 JSON 을 대신 만들어 줍니다.**

In [ ]:
import json

# tool_call_schema : 모델이 이 도구를 부를 때 채워야 할 인자들의 JSON 명세
spec = supply_stock.tool_call_schema.model_json_schema()

# ensure_ascii=False : 한글이 \uXXXX 로 깨져 보이지 않게 한다
print(json.dumps(spec, ensure_ascii=False, indent=2))

> `description` 은 **docstring** 에서, `properties` 의 타입(`"type": "string"`)은 **타입힌트**에서 왔습니다. **docstring 과 타입힌트를 대충 적으면 이 JSON 이 부실해지고, 그대로 모델의 판단이 나빠집니다.**

## 도구는 그냥 함수다 — 직접 불러 보기

여기까지 **모델은 한 번도 등장하지 않았습니다.** 도구는 우리가 직접 부를 수도 있습니다. 인자는 `invoke` 에 **딕셔너리**로 넘깁니다.

In [ ]:
# 도구를 직접 호출 — 모델 없이도 그냥 함수처럼 동작한다는 것을 확인합니다.
print(supply_stock.invoke({'item_name': '무선 마우스'}))     # 40 이 나온다
print(restock_cost.invoke({'item_name': '보안 USB', 'count': 3}))   # 32000 * 3
print('없는 비품:', supply_stock.invoke({'item_name': '3D 프린터'}))   # 사전에 없으면 0

## ⚖️ 구조화된 출력 vs 도구 — 헷갈리기 쉬운 한 쌍

둘 다 **스키마**를 적어 넘기기 때문에 헷갈립니다. 판별 질문은 하나입니다 — **"이 답을 만들려면 우리 데이터를 뒤져야 하나?"**

- **그렇다 → 도구.** ("보안 USB 재고가 몇 개야?" — 모델은 우리 창고를 모릅니다.)
- **아니다 → 구조화된 출력.** ("이 문의는 어느 갈래야?" — 문의 본문만 있으면 모델이 판단합니다.)

<img src="images/구조화출력_vs_도구.png" width="820">

*같은 스키마라도 목적이 다릅니다 — 바깥 정보가 필요하면 도구, 답의 모양만 정하면 구조화된 출력입니다.*

둘은 **함께** 쓰입니다. 도구로 사내 데이터를 가져온 뒤 그 결과를 구조화된 출력으로 담는 것이 흔한 흐름입니다.

### 🖐️ 함께 따라하기 — 구내식당 메뉴 도구 만들기

**다른 도메인**으로 도구를 하나 만들어 봅니다. 요일을 받아 구내식당 메뉴를 알려 주는 도구입니다.

1. 아래 사전을 만듭니다.
   `_MENU = {'월': '제육볶음', '화': '김치찌개', '수': '돈가스', '목': '비빔밥', '금': '해물순두부'}`
2. `@tool` 을 붙인 `cafeteria_menu(day_name: str) -> str` 함수를 정의합니다.
   docstring 은 **모델이 읽는 명세**라는 점을 기억하고, 무엇을 받아 무엇을 돌려주는지 정확히 적습니다.
   목록에 없는 요일이면 `'그날은 메뉴가 없습니다.'` 를 돌려줍니다.
3. `.name`·`.description`·`.args` 를 출력합니다.
4. `cafeteria_menu.invoke({'day_name': '수'})` 로 직접 호출해 결과를 출력합니다.

**확인 기준**: `.description` 에 내가 쓴 docstring 이 그대로 들어 있다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) _MENU 사전을 만든다
# 2) @tool 을 붙인 cafeteria_menu(day_name: str) -> str 를 정의한다(docstring 을 정확히)
# 3) .name / .description / .args 를 출력한다
# 4) cafeteria_menu.invoke({'day_name': '수'}) 를 출력한다

### ✅ 바로 확인 퀴즈

**1.** 모델은 여러 도구 중 **무엇을 보고** 어떤 도구를 쓸지 판단하나요?

<details><summary>정답 보기</summary>

도구의 **이름과 docstring(설명)**, 그리고 **인자 명세**를 보고 판단합니다. 모델은 함수 본문을 보지 못합니다.

</details>

**2.** `@tool` 을 붙인 함수를 직접 실행하려면 어떻게 인자를 넘기나요?

<details><summary>정답 보기</summary>

`도구.invoke({'인자이름': 값, ...})` 처럼 **딕셔너리**로 넘깁니다(예: `restock_cost.invoke({'item_name': '보안 USB', 'count': 3})`).

</details>

**3.** "이 리뷰가 긍정인지 부정인지 알려줘"는 도구와 구조화된 출력 중 어느 쪽인가요?

<details><summary>정답 보기</summary>

**구조화된 출력**입니다. 판단에 필요한 것은 리뷰 본문뿐이고 **우리 데이터를 뒤질 필요가 없기** 때문입니다. 반대로 "별점 5점 리뷰가 몇 개야?"는 우리 데이터를 뒤져야 하므로 **도구**입니다.

</details>

---
# 3. 에이전트에 붙이기 — `create_agent` 와 메시지 궤적

## 왜 이렇게 할까요?
도구를 만들었어도, **언제 어떤 도구를 부를지** 누군가는 정해야 합니다. 그 판단을 모델에게 맡긴 것이 **에이전트**입니다.

## 비유
- **체인** = 정해진 요리법. 재료를 넣으면 늘 같은 순서로 요리합니다.
- **에이전트** = 요리사. 상황을 보고 **어떤 도구(칼·냄비)를 쓸지 스스로** 정합니다.

## 문법
```python
agent = create_agent(model, [도구1, 도구2])
result = agent.invoke({'messages': '질문 문자열'})
```

`messages` 에는 **질문 문자열 하나**를 그대로 넣으면 됩니다. 결과는 딕셔너리이고, 그 안의 `result['messages']` 에 **오간 메시지가 순서대로** 쌓입니다.

## 메시지 궤적 — 무슨 일이 있었는지가 다 남는다

| 순서 | 메시지 | 뜻 |
|---|---|---|
| 1 | **HumanMessage** | 사용자의 질문 |
| 2 | **AIMessage** (`.tool_calls` 있음) | 모델이 "이 도구를 이런 인자로 부르자"고 **결정** |
| 3 | **ToolMessage** | 도구를 실제로 실행한 **결과** |
| 4 | **AIMessage** (최종) | 도구 결과를 반영한 **자연어 답변** |

<img src="images/에이전트_메시지_궤적.png" width="820">

*질문 하나에 네 개의 메시지가 남습니다 — 어디서 무엇이 결정됐는지 되짚을 수 있습니다.*

> 최종 답 텍스트는 `result['messages'][-1].text` 로 얻습니다(`.content` 는 여러 조각의 리스트일 수 있어 `.text` 가 안전합니다 — 지난 시간에 배운 그 관례 그대로입니다).

In [ ]:
# 도구 두 개를 쥔 에이전트를 만들고 질문 하나를 던집니다.
from langchain.agents import create_agent
from langchain_core.messages import ToolMessage

agent = create_agent(model, [supply_stock, restock_cost])
result = agent.invoke({'messages': '무선 마우스 창고에 몇 개 남았어?'})   # messages 에 질문 문자열 하나

# 궤적을 순서대로 훑어봅니다 — 도구 호출 결정은 .tool_calls 에 들어 있습니다.
for m in result['messages']:
    kind = type(m).__name__
    if getattr(m, 'tool_calls', None):
        print(kind, '-> 도구호출:', [(c['name'], c['args']) for c in m.tool_calls])
    else:
        print(kind, '->', m.text if m.text else '(내용 없음)')

**우리가 루프를 돌리지 않았는데 루프가 돌았습니다.** OpenAI SDK 를 직접 쓰던 단원에서는 (1) 모델에게 묻고 (2) 어떤 함수를 부르려 하는지 읽고 (3) 우리가 직접 그 함수를 실행하고 (4) 결과를 다시 모델에 넣어 주는 네 단계를 **손으로** 짰습니다. `create_agent` 는 그 네 단계를 대신 돌아 줍니다 — 우리는 **도구만 잘 만들면** 됩니다.

In [ ]:
# 궤적에서 필요한 부분만 꺼내 쓰는 두 가지 관용구
tool_msgs = [m for m in result['messages'] if isinstance(m, ToolMessage)]
print('불린 도구:', [(m.name, m.content) for m in tool_msgs])
print('최종 답  :', result['messages'][-1].text)

## 역할 정해 주기 — `system_prompt`

`create_agent` 에 **`system_prompt`** 를 주면 에이전트의 **역할·말투**를 정할 수 있습니다. 역할을 줘도 **도구 선택은 그대로**이고, 주로 **답의 표현**이 달라집니다.

In [ ]:
# 도구는 똑같고 역할만 준다 — 무엇이 바뀌고 무엇이 안 바뀌는지 비교해 보세요.
polite_agent = create_agent(
    model, [supply_stock, restock_cost],
    system_prompt='너는 친절한 사내 헬프데스크 상담원이다. 항상 정중한 존댓말로 답한다.')
res_role = polite_agent.invoke({'messages': '무선 마우스 창고에 몇 개 남았어?'})
print('불린 도구:', [m.name for m in res_role['messages'] if isinstance(m, ToolMessage)])
print('최종 답  :', res_role['messages'][-1].text)

## 도구가 필요 없는 질문

에이전트의 미덕은 **필요할 때만 도구를 쓰는 것**입니다. 도구는 그대로 붙어 있고 질문만 바꿔 봅니다.

In [ ]:
# 도구는 그대로 두 개 붙어 있다 — 달라지는 것은 질문뿐이다.
res_notool = agent.invoke({'messages': '헬프데스크가 하는 일이 뭐야? 한 문장으로 알려줘.'})
used = [m.name for m in res_notool['messages'] if isinstance(m, ToolMessage)]
print('도구 호출 수:', len(used))
print('최종 답     :', res_notool['messages'][-1].text)

## 도구도 쓰고, 답도 스키마로 — `response_format`

여기까지 오면 1절과 3절이 각각 반쪽입니다.

- 1절의 `with_structured_output` 은 답을 **칸에 담아** 주지만 **도구를 쓰지 못합니다** — 모델이 이미 아는 것만 정리합니다.
- 3절의 에이전트는 **도구를 쓰지만** 마지막 답이 **자유 문장**입니다 — 그대로는 표에 못 넣습니다.

실무에서 필요한 것은 대개 **둘 다**입니다. "도구로 재고를 조회하고, 그 결과를 정형화된 티켓으로 정리해서 돌려줘" 같은 요구지요. 이럴 때 `create_agent` 에 **`response_format`** 을 줍니다.

## 문법 — `response_format=ProviderStrategy(스키마, strict=True)`

| | `model.with_structured_output(S)` (1절) | `create_agent(..., response_format=ProviderStrategy(S))` |
|---|---|---|
| 스키마를 씌우는 대상 | **모델 하나** | **에이전트 전체** |
| 도구 | 쓰지 않는다 | **도구를 다 쓴 뒤** 마지막 답을 정형화한다 |
| 결과를 꺼내는 곳 | `invoke(...)` 의 반환값이 곧 그 객체 | **`res['structured_response']`** |
| 궤적 | 남지 않는다 | `res['messages']` 에 **그대로 남는다** |

`ProviderStrategy` 는 "이 정형화를 **모델 공급자에게 맡기겠다**"는 뜻입니다(`from langchain.agents.structured_output import ProviderStrategy`). 결과 딕셔너리에 **키가 하나 더 생기는 것**이 핵심입니다 — 궤적은 그대로 두고 정형 결과만 얹어 줍니다.

In [ ]:
# 도구를 쓴 뒤 그 결과를 스키마에 담아 돌려받습니다 — 궤적과 정형 결과를 한 번에 얻습니다.
from langchain.agents.structured_output import ProviderStrategy


class StockReport(BaseModel):
    """재고 문의 한 건을 처리한 결과."""

    item_name: str = Field(description='문의한 비품 이름')
    # 이 칸은 모델이 지어내는 값이 아니라 '도구가 알려 준 값'이다 - 그래서 도구가 반드시 필요하다.
    stock: int = Field(description='도구가 알려 준 현재 재고 수량')
    status: Literal['충분', '부족', '없음'] = Field(
        description='재고 20개 이상이면 충분, 1~19개면 부족, 0개면 없음')
    note: str = Field(description='담당자에게 남길 한 문장')


# strict=True : 스키마를 '부탁'이 아니라 '규칙'으로 강제한다(바로 아래 셀에서 있고 없고를 비교한다).
report_agent = create_agent(model, [supply_stock, restock_cost],
                            response_format=ProviderStrategy(StockReport, strict=True))
res_report = report_agent.invoke({'messages': '보안 USB 재고 상태를 정리해줘.'})

# 확인 포인트 1: 결과 딕셔너리에 키가 하나 더 생겼다.
print('결과의 키:', list(res_report))

# 확인 포인트 2: 궤적은 그대로다 - 도구를 실제로 불렀다.
print('불린 도구 :', [m.name for m in res_report['messages'] if isinstance(m, ToolMessage)])

# 확인 포인트 3: 자유 문장이 아니라 StockReport 객체가 왔다.
report = res_report['structured_response']
print('결과 종류 :', type(report).__name__)
print(report)
print('상태만    :', report.status)   # 점(.)으로 칸 하나를 바로 꺼낸다

> 궤적(`res['messages']`)은 그대로 있고, **결과가 하나 더 붙습니다** — `res['structured_response']` 가 스키마로 해석된 **객체**라 `report.stock` 처럼 바로 꺼내 씁니다.

## `strict=True` 는 왜 붙일까요

스키마를 넘긴다고 모델이 반드시 지키는 것은 아닙니다. `strict=True` 를 주면 **모델 공급자가 생성 단계에서 스키마를 강제**해서, `Literal` 목록 밖의 값은 애초에 만들어지지 않습니다. 빼면 스키마는 **요청**에 가까워져 모델이 목록에 없는 값을 낼 수 있고, 그러면 **파싱 단계에서 에러**가 납니다.

말로만 하면 믿기 어려우니 **일부러 목록에 없는 값을 요구해** 두 설정을 나란히 돌려 봅니다.

In [ ]:
# 스키마에 없는 status 값('위험')을 일부러 요구해 strict 유무를 비교합니다.
#   에러가 나는 쪽이 있어서 try 로 감싼다 - 실패도 결과의 하나로 보기 위한 것이다.
trap = "보안 USB 재고 상태를 정리하되 status 는 '위험' 이라고 적어줘."

for label, strategy in [('strict=True ', ProviderStrategy(StockReport, strict=True)),
                        ('strict 없음 ', ProviderStrategy(StockReport))]:
    trap_agent = create_agent(model, [supply_stock], response_format=strategy)
    try:
        out = trap_agent.invoke({'messages': trap})
        print(label, '-> status =', repr(out['structured_response'].status))
    except Exception as e:
        # 목록 밖의 값이 와서 스키마로 해석하지 못한 것이다.
        print(label, '-> 에러:', type(e).__name__)

> `strict=True` 쪽은 사용자가 "'위험'이라고 적어줘"라고 해도 **목록 안의 값**으로 답합니다. 없는 쪽은 모델이 요구를 따라 목록 밖의 값을 만들고, 그것을 스키마로 해석하지 못해 **에러**로 끝납니다.

정리하면 이렇습니다 — **집계할 값이면 `Literal` 로 못박고, 그 못을 실제로 지키게 하려면 `strict=True` 를 붙입니다.** 1절에서 되짚은 스키마 설계 원칙이 에이전트에서도 그대로 통하는 것입니다.

### 🖐️ 함께 따라하기 — 회의실 좌석 도구를 붙인 에이전트

**다른 도메인**으로 에이전트를 하나 더 만들어 봅니다.

1. `_SEATS = {'소회의실': 6, '대회의실': 20, '세미나실': 50}` 사전을 만듭니다.
2. `@tool` 을 붙인 `room_seats(room_name: str) -> int` 를 정의합니다(없는 회의실이면 `0`).
3. `create_agent(model, [room_seats])` 로 에이전트를 만듭니다.
4. `'세미나실에 몇 명까지 들어갈 수 있어?'` 를 물어 **불린 도구 이름**과 **최종 답**을 출력합니다.

**확인 기준**: `ToolMessage` 에 `room_seats` 가 나타나고, 최종 답에 50이라는 수가 들어간다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) _SEATS 사전을 만든다
# 2) @tool 을 붙인 room_seats(room_name: str) -> int 를 정의한다
# 3) create_agent(model, [room_seats]) 로 에이전트를 만든다
# 4) '세미나실에 몇 명까지 들어갈 수 있어?' 를 물어 불린 도구와 최종 답을 출력한다

### ✅ 바로 확인 퀴즈

**1.** `result['messages']` 에서 **도구를 실제로 실행한 결과**가 담긴 메시지 종류는?

<details><summary>정답 보기</summary>

**ToolMessage** 입니다. 그 앞의 **AIMessage** 에는 모델이 도구를 **부르기로 한 결정**(`.tool_calls`)이 담깁니다.

</details>

**2.** 도구가 필요 없는 질문이면 궤적이 어떻게 달라지나요?

<details><summary>정답 보기</summary>

**ToolMessage 가 나타나지 않습니다.** 모델이 도구를 부를 필요가 없다고 판단해 바로 최종 답을 냅니다.

</details>

**3.** `system_prompt` 를 주면 무엇이 바뀌고 무엇이 그대로인가요?

<details><summary>정답 보기</summary>

**답의 말투·태도**가 바뀝니다. **어떤 도구를 부를지에 대한 판단은 그대로**입니다 — 그 판단은 질문과 도구 설명에서 나오기 때문입니다.

</details>

**4.** 도구로 재고를 조회한 **뒤** 그 결과를 표에 넣을 수 있는 형태로 받고 싶습니다. 1절의 `with_structured_output` 으로 될까요?

<details><summary>정답 보기</summary>

안 됩니다. `with_structured_output` 은 **모델 하나**를 감싸는 것이라 **도구를 쓰지 못합니다**. 도구를 쓰면서 답도 정형화하려면 `create_agent(..., response_format=ProviderStrategy(스키마, strict=True))` 를 쓰고 결과를 **`res['structured_response']`** 에서 꺼냅니다. 궤적은 `res['messages']` 에 그대로 남습니다.

</details>

---
# 4. 실무에서 쓸 도구 만들기

## 왜 이 절이 필요할까요?
앞에서 만든 도구는 사전에서 값을 꺼내는 것이 전부였습니다. 동작은 하지만 실무에 그대로 두면 두 가지가 걸립니다.

- **예외를 던지면 대화가 거기서 멈춥니다.** 사전에 없는 이름이 하나 들어온 것뿐인데 에이전트가 다시 시도할 기회를 잃습니다.
- **도구가 할 수 있는 일이 넓으면 모델의 판단 실수가 그대로 사고가 됩니다.**

이 둘을 어떻게 막는지 보고, 사내 데이터 조회·규칙 계산·외부 API 세 가지 **실무형 도구**를 만들어 봅니다. 만든 도구를 에이전트가 **잘 고르게 하는** 문제는 도구가 여러 개일 때 드러나므로 **다음 시간**에 다룹니다.

## 실패는 문자열로 돌려준다

도구는 **실패합니다.** 없는 번호, 잘못된 형식, 끊긴 네트워크. 이때 **예외를 던지면 에이전트가 그 자리에서 멈춥니다.** 반대로 실패를 **설명이 담긴 문자열**로 돌려주면, 그 문자열이 `ToolMessage` 로 모델에게 전달되고 모델은 그것을 **읽고** 고쳐 다시 시도하거나 사용자에게 알맞게 안내합니다.

자산 관리번호로 담당자를 찾는 도구를 만듭니다. 없는 번호가 오면 **에러를 내지 않고**, 무엇이 잘못됐고 형식이 어떤지를 **글로** 돌려줍니다.

In [ ]:
# 실패를 문자열로 돌려주는 도구 - 없는 번호가 와도 예외를 던지지 않습니다.
_OWNER = {'IT-2043': '김서연 / 개발팀', 'IT-1188': '박도윤 / 총무팀'}


@tool
def asset_owner(asset_tag: str) -> str:
    """자산 관리번호(asset_tag, 'IT-####' 형식)로 그 장비를 쓰는 담당자와 부서를 찾는다."""
    tag = asset_tag.strip().upper()
    if tag not in _OWNER:
        # 없는 번호는 에러가 아니라 '무엇이 잘못됐고 어떻게 고치면 되는지'를 담은 글로 돌려준다.
        return f"찾을 수 없음: {tag}. 관리번호는 'IT-' 뒤에 숫자 네 자리입니다(예: IT-2043)."
    return _OWNER[tag]


clear_agent = create_agent(model, [asset_owner])

print(asset_owner.invoke({'asset_tag': 'IT-2043'}))

같은 조회를 **예외로 실패하게** 만든 도구와 나란히 놓고 봅니다.

In [ ]:
# 예외를 던지는 도구와 문자열을 돌려주는 도구를 나란히 비교합니다.
@tool
def asset_owner_strict(asset_tag: str) -> str:
    """자산 관리번호(asset_tag)로 담당자를 찾는다. 없으면 에러가 난다."""
    return _OWNER[asset_tag.strip().upper()]        # 없는 열쇠면 KeyError 가 터진다


# 없는 관리번호를 넣어 두 도구의 반응을 나란히 봅니다.
try:
    asset_owner_strict.invoke({'asset_tag': 'IT-9999'})
except Exception as e:
    print('예외를 던지는 도구 ->', type(e).__name__, e)

print('문자열을 돌려주는 도구 ->', asset_owner.invoke({'asset_tag': 'IT-9999'}))

이제 문자열을 돌려주는 도구를 붙인 에이전트에 **없는 관리번호**를 물어봅니다. 에러로 멈추는 대신 무슨 일이 일어나는지 보세요.

In [ ]:
# 도구가 돌려준 안내 문자열이 ToolMessage 로 모델에게 전달됩니다.
res_fail = clear_agent.invoke({'messages': 'IT-9999 장비 담당자가 누구야?'})
print('도구가 돌려준 값:', [m.content for m in res_fail['messages']
                          if isinstance(m, ToolMessage)])
print('최종 답        :', res_fail['messages'][-1].text)

> 대화가 **끊기지 않았습니다.** 모델이 도구의 안내를 읽고 사용자에게 "그 번호는 없다, 형식은 이렇다"고 설명해 주었습니다. 실패 메시지를 **사람이 읽어도 말이 되게** 쓰면, 그대로 모델의 다음 행동 지침이 됩니다.

## 할 수 있는 일을 좁게

도구는 모델이 **직접 실행하는 코드**입니다. 그러니 도구가 할 수 있는 일이 곧 **모델에게 준 권한**입니다.

- **읽기 전용으로 만든다** — 조회 도구는 조회만. 지우거나 고치는 일은 도구로 열지 않습니다.
- **상한선을 도구 안에 못박는다** — "한 번에 최대 20건"처럼 한계를 함수 안에 둡니다. 프롬프트로 부탁하는 것과 달리, 코드에 박아 두면 **반드시** 지켜집니다.
- **위험한 일은 사람의 확인을 거친다** — 실제로 주문을 넣거나 메일을 보내는 도구라면, 도구가 하는 일은 "신청서를 만들어 두는 것"까지로 두고 최종 실행은 사람이 누르게 합니다.

> 모델이 늘 옳은 판단을 한다고 가정하지 마세요. **도구가 할 수 없는 일은 모델도 할 수 없습니다** — 이것이 가장 확실한 안전장치입니다.

## 실무형 도구 세 가지

이제 위의 두 가지를 적용해 **실제로 쓸 만한 도구** 세 가지를 만듭니다.

| 종류 | 무엇을 하나 | 왜 도구여야 하나 |
|---|---|---|
| **사내 데이터 조회** | 우리 CSV·데이터베이스를 뒤져 답을 만든다 | 모델은 우리 회사 문서를 본 적이 없다 |
| **규칙 계산** | 회사 규정이 정한 계산을 코드로 한다 | 모델의 산수는 틀릴 수 있고, 규정은 **정확해야** 한다 |
| **외부 REST API** | 인터넷 서비스에 요청을 보내 최신 값을 받아 온다 | 오늘의 환율은 학습 이후의 일이다 |

### (1) 사내 데이터 조회
**판다스 단원**에서 배운 `read_csv` 와 조건 필터를 도구 안에 넣은 것뿐입니다. 여기에는 두 가지가 함께 들어가 있습니다 — **받은 인자를 도구가 다듬는 것**(문장이 와도 첫 낱말만 쓴다)과 **실패를 문자열로 돌려주는 것**입니다. 앞의 것은 코드 한 줄이라 지나치기 쉬운데, 다음 시간에 이것이 왜 중요한지를 증상과 함께 다시 봅니다.

In [ ]:
# 실무형 도구 (1) 사내 데이터 조회 - 우리 CSV 를 뒤져 답을 만들어 주는 도구입니다.
import pandas as pd

faq_df = pd.read_csv('data/helpdesk_faq.csv')


@tool
def faq_answer(keyword: str) -> str:
    """사내 헬프데스크 FAQ 에서 안내문을 찾아 제목과 본문을 돌려준다.
    keyword 는 'VPN'·'비밀번호'·'프린터' 처럼 낱말 하나로 넘긴다."""
    word = keyword.strip().split()[0]      # 문장이 와도 첫 낱말만 쓴다(인자 다듬기)
    # 제목에서 먼저 찾고, 없을 때만 본문에서 찾는다 - 제목이 더 정확한 신호이기 때문이다.
    hit = faq_df[faq_df['title'].str.contains(word)]
    if hit.empty:
        hit = faq_df[faq_df['text'].str.contains(word)]
    if hit.empty:
        return f"'{word}' 안내문이 없습니다. 'VPN'·'비밀번호'·'프린터' 같은 낱말로 다시 찾아보세요."
    row = hit.iloc[0]
    return f"[{row['title']}] {row['text']}"


print(faq_answer.invoke({'keyword': '비밀번호 재설정'}))   # 문장을 넣어도 '비밀번호'로 다듬어 찾는다

### (2) 규칙 계산

"5만원 이상이면 배송비 무료" 같은 규정은 **모델에게 물어볼 일이 아닙니다.** 규정은 회사가 정한 것이고, 코드가 계산하면 **언제나 같은 답**이 나옵니다. 모델이 하는 일은 "이 계산이 필요하다"고 판단해 **금액과 조건을 넘겨 주는 것**까지입니다.

In [ ]:
# 실무형 도구 (2) 규칙 계산 - 회사 규정이 정한 계산은 코드가 해야 합니다.
@tool
def delivery_fee(total_price: int, is_express: bool) -> str:
    """주문 금액(total_price, 원)과 빠른배송 여부(is_express)로 배송비를 계산한다.
    5만원 이상이면 기본 배송비가 무료이고, 빠른배송이면 3000원이 더 붙는다."""
    base = 0 if total_price >= 50000 else 3000
    express = 3000 if is_express else 0
    return f'기본 {base}원 + 빠른배송 {express}원 = 총 {base + express}원'


# 규칙이 코드에 있으니 결과가 늘 같다 - 같은 입력이면 몇 번을 불러도 같은 답이 나온다.
print('4만원 일반배송 :', delivery_fee.invoke({'total_price': 40000, 'is_express': False}))
print('6만원 빠른배송 :', delivery_fee.invoke({'total_price': 60000, 'is_express': True}))

### (3) 외부 REST API

**데이터 수집 단원**에서 배운 `requests` 를 그대로 도구 안에 넣습니다 — 이미 아는 API 호출을 함수로 감싸고 `@tool` 을 붙였을 뿐입니다. 다만 여기에 **실패를 문자열로 돌려주는 처리가 반드시 필요합니다.** 인터넷은 언제든 끊기고, 서비스는 언제든 점검에 들어갑니다.

- `raise_for_status()` 는 응답이 4xx·5xx 면 **예외**를 냅니다. 그대로 두면 그 예외가 도구 밖으로 튀어나가 **에이전트가 멈춥니다.**
- 그래서 요청 전체를 `try` 로 감싸고, 실패하면 **사람이 읽어도 말이 되는 문자열**을 돌려줍니다. 그러면 모델이 그것을 읽고 사용자에게 안내합니다.

> 여기서 쓰는 환율 서비스는 키가 필요 없는 공개 API 입니다. 환율은 매일 바뀌므로 **결과 값을 외우지 마세요** — 확인할 것은 "모델이 혼자서는 절대 알 수 없는 값을 도구가 가져왔다"는 사실입니다.

In [ ]:
# 외부 REST API 를 도구로 — 인터넷의 환율 서비스에 실제로 요청을 보냅니다(데이터 수집 단원의 requests 그대로).
import requests


@tool
def usd_krw_rate() -> str:
    """가장 최근 고시된 원/달러 환율을 외부 환율 API 에서 받아 온다. 1달러가 몇 원인지 돌려준다."""
    try:
        res = requests.get('https://api.frankfurter.dev/v1/latest',
                           params={'from': 'USD', 'to': 'KRW'}, timeout=10)
        res.raise_for_status()      # 4xx·5xx 면 예외를 낸다(조용한 실패를 만들지 않는다)
        data = res.json()           # 응답은 JSON — 필요한 값만 꺼낸다
    except requests.RequestException as e:
        # 도구 설계 원칙 3 — 실패도 '문자열'로 돌려준다.
        #   예외를 밖으로 던지면 에이전트가 그 자리에서 멈추지만, 문자열이면 모델이 그것을 읽고
        #   사용자에게 안내하거나 다시 시도한다. 인터넷은 언제든 끊길 수 있다.
        return f'환율을 가져오지 못했습니다: {e}. 잠시 뒤 다시 시도해 주세요.'
    return f"{data['date']} 기준 1 USD = {data['rates']['KRW']} KRW"


print(usd_krw_rate.invoke({}))     # 도구를 직접 불러 응답을 눈으로 확인(인자가 없어 빈 딕셔너리)

이제 세 도구를 한 에이전트에 붙이고, **모델이 모르는 오늘 환율**을 물어봅니다.

In [ ]:
# 성격이 다른 도구 세 개를 한 에이전트에 붙인다 — 모델이 알맞은 것을 고른다.
office_agent = create_agent(model, [faq_answer, delivery_fee, usd_krw_rate])
res_fx = office_agent.invoke({'messages': '오늘 1달러는 몇 원이야?'})
print('불린 도구:', [m.name for m in res_fx['messages'] if isinstance(m, ToolMessage)])
print('최종 답  :', res_fx['messages'][-1].text)

> 2절 도입부에서 "모델은 오늘 환율을 모른다"고 했습니다. 이제 답합니다 — 모델이 똑똑해진 것이 아니라 **바깥을 볼 수 있는 손**이 생긴 것입니다.

### 🖐️ 함께 따라하기 — 주차 등록 도구 만들기

**다른 도메인**으로, 인자 다듬기와 실패 처리를 한 번에 적용한 도구를 만들어 봅니다.

1. `@tool` 을 붙인 `parking_register(car_number: str) -> str` 를 정의합니다.
2. **인자 다듬기**: 사람은 `'12가 3456'`·`'12가-3456'` 처럼 다양하게 적습니다. 공백과 하이픈(`-`)을 지워 `'12가3456'` 형태로 맞춥니다.
3. **실패는 문자열로**: 다듬은 번호가 **7자리**가 아니면 `"번호 형식이 올바르지 않습니다. '12가3456' 처럼 입력하세요."` 를 돌려줍니다.
4. 정상이면 `f'{다듬은번호} 방문 주차 등록 완료(당일 4시간 무료)'` 를 돌려줍니다.
5. `'12가 3456'`, `'12가-3456'`, `'1234'` 세 가지로 직접 호출해 결과를 출력합니다.

**확인 기준**: 앞의 두 입력은 **같은 결과**가 나오고, 세 번째는 에러로 멈추지 않고 **안내 문자열**이 나온다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) @tool 을 붙인 parking_register(car_number: str) -> str 를 정의한다
# 2) 공백과 하이픈을 지워 번호를 다듬는다
# 3) 다듬은 번호가 7자리가 아니면 안내 문자열을 돌려준다
# 4) 정상이면 등록 완료 문장을 돌려준다
# 5) '12가 3456' / '12가-3456' / '1234' 세 가지로 직접 호출해 출력한다

### ✅ 바로 확인 퀴즈

**1.** 도구가 실패했을 때 예외를 던지지 않고 문자열을 돌려주는 이유는?

<details><summary>정답 보기</summary>

예외를 던지면 **에이전트가 그 자리에서 멈춥니다.** 문자열로 돌려주면 그 내용이 `ToolMessage` 로 모델에게 전달되어, 모델이 **읽고 고쳐 다시 시도**하거나 사용자에게 알맞게 안내할 수 있습니다.

</details>

**2.** 같은 모델·같은 질문인데 도구 선택이 흔들린다면 먼저 무엇을 의심해야 하나요?

<details><summary>정답 보기</summary>

**도구의 이름과 docstring** 입니다. `temperature` 를 낮춰도 판단 근거는 생기지 않습니다 — 근거는 우리가 쓴 설명에서만 나옵니다.

</details>

**3.** "한 번에 최대 20건만 조회"라는 제한을 프롬프트가 아니라 **도구 함수 안에** 두는 것이 나은 이유는?

<details><summary>정답 보기</summary>

프롬프트는 **부탁**이라 지켜지지 않을 수 있지만, 코드에 박아 두면 **반드시** 지켜지기 때문입니다. 도구가 할 수 없는 일은 모델도 할 수 없습니다.

</details>

---
# 5. MCP — 남이 만든 도구 서버에 붙기

> **이 절의 범위**: 서버를 직접 만드는 것은 이 단원의 범위가 아닙니다 — 오늘은 **이미 있는 서버에 붙이는 법**만 배웁니다.

## 왜 표준이 필요할까요?
지금까지 만든 `@tool` 은 **우리 노트북 안에서만** 삽니다. 그런데 실무에서는

- 회사의 다른 팀이 만든 **사내 시스템 조회 도구**를 쓰고 싶고,
- 공개된 **파일 시스템·깃허브·데이터베이스 도구**를 그대로 가져다 쓰고 싶고,
- 우리가 만든 도구를 **다른 프로그램**(다른 에이전트, 챗 앱)에서도 쓰게 하고 싶습니다.

그러려면 "도구를 이렇게 알리고 이렇게 부른다"는 **약속된 규격**이 필요합니다. 그 규격이 **MCP(Model Context Protocol)** 입니다.

## 비유 — USB-C
예전에는 기기마다 충전 단자가 달라 케이블을 여러 개 들고 다녔습니다. USB-C 는 **한 규격**이라 어느 회사 기기든 같은 구멍에 꽂힙니다. MCP 도 마찬가지입니다 — 누가 만든 도구든 **같은 방식으로** 목록을 받아 **같은 방식으로** 부릅니다.

<img src="images/mcp_usb_비유.png" width="820">

*도구를 가진 쪽이 **서버**, 그 도구를 받아 쓰는 우리 노트북이 **클라이언트**입니다.*

## 오늘 붙어 볼 서버
`data/mcp_server.py` 에 **환율 변환**·**남은 일수 계산**·**신간 도서 검색** 도구를 가진 MCP 서버를 만들어 두었습니다. 우리 노트북은 이 서버를 **별도 프로세스로 띄우고** 도구 목록을 받아 옵니다.

## 왜 비동기(async)인가
MCP 클라이언트는 **다른 프로세스나 다른 컴퓨터의 서버**와 이야기합니다. 그 응답을 기다리는 동안 프로그램 전체가 멈춰 있으면 낭비이므로, MCP 는 **기다리는 동안 다른 일을 할 수 있는 방식(비동기)** 으로 설계돼 있습니다. 그래서 도구 목록을 받아 올 때도, 에이전트를 실행할 때도 **`asyncio.run(...)`** 으로 감싸 부릅니다 (`invoke` 대신 **`ainvoke`**). 아래 준비 셀은 주피터 안에서도 이 방식이 되게 해 줍니다 — **내용은 이해하지 않아도 됩니다.**

In [ ]:
# [제공 코드] 비동기 실행 준비 — MCP 도구는 비동기(async)라 이 셀이 필요합니다(실행만 하세요).
import asyncio

import nest_asyncio

nest_asyncio.apply()   # Jupyter 안에서도 asyncio.run(...) 을 쓸 수 있게 해 준다


In [ ]:
# [제공 코드] MCP 서버를 서브프로세스로 띄우고 도구 목록을 가져옵니다.
import sys
from pathlib import Path
from langchain_mcp_adapters.client import MultiServerMCPClient

# transport='stdio' : 서버를 별도 프로세스로 띄우고 표준입출력으로 이야기한다는 뜻입니다.
server_path = str((Path('data') if Path('data').exists() else Path('../data')) / 'mcp_server.py')
mcp_client = MultiServerMCPClient({
    'utils': {'transport': 'stdio', 'command': sys.executable, 'args': [server_path]},
})
mcp_tools = asyncio.run(mcp_client.get_tools())   # 서버가 어떤 도구를 갖고 있는지 물어봅니다
print('MCP 도구:', [t.name for t in mcp_tools])

받아 온 도구들이 어떻게 생겼는지 봅니다. **우리가 만든 `@tool` 과 똑같은 모양**입니다.

In [ ]:
# 이름과 설명 — 2절에서 본 우리 도구의 .name / .description 과 같은 구조입니다.
for t in mcp_tools:
    print('name       :', t.name)
    print('description:', t.description.splitlines()[0])
    print('args       :', t.args)
    print()

> 우리가 만든 도구인지, 남이 만들어 서버로 띄운 도구인지 **모델은 구분하지 않습니다.** 이름·설명·인자 명세만 같은 모양이면 되기 때문입니다. 이것이 표준 규격의 힘입니다.

받아 온 도구 리스트를 **그대로** `create_agent` 에 넘깁니다. 코드가 달라지는 부분은 `invoke` → `asyncio.run(... .ainvoke(...))` 하나뿐입니다.

In [ ]:
# MCP 도구는 비동기라 ainvoke 를 쓰고, 그것을 asyncio.run 으로 감싸 실행합니다.
mcp_agent = create_agent(model, mcp_tools)
res_mcp = asyncio.run(mcp_agent.ainvoke({'messages': '13000원은 몇 달러야?'}))

# MCP 도구의 결과는 조각 리스트일 수 있어 str() 로 감싸 출력합니다.
print('불린 도구:', [(m.name, str(m.content)) for m in res_mcp['messages']
                    if isinstance(m, ToolMessage)])
print('최종 답  :', res_mcp['messages'][-1].text)

## 우리 도구와 MCP 도구를 섞기

가장 중요한 장면입니다. 우리가 2절에서 만든 `supply_stock` 과 서버가 준 도구를 **하나의 리스트로 합쳐** 에이전트에 넘깁니다. 모델은 둘을 구분하지 않고 **필요한 것을 골라** 씁니다.

In [ ]:
# 리스트 두 개를 더하기(+)로 합칠 뿐입니다 — 출신이 다른 도구가 한 상자에 들어갑니다.
mixed_agent = create_agent(model, mcp_tools + [supply_stock])
res_mix = asyncio.run(mixed_agent.ainvoke({'messages': '보안 USB 재고가 몇 개고, 32000원은 몇 달러야?'}))

# 확인 포인트: 질문 하나에 '우리 도구'와 'MCP 도구'가 모두 불렸는지.
print('불린 도구:', [(m.name, str(m.content)) for m in res_mix['messages']
                    if isinstance(m, ToolMessage)])
print('최종 답  :', res_mix['messages'][-1].text)

> 질문 하나에 **우리 창고 도구**와 **서버의 환율 도구**가 함께 불렸습니다. 도구가 어디에 살든 에이전트에게는 똑같은 도구입니다.

**어디서 서버를 구하나요?** 공개된 MCP 서버 목록이 한곳에 모여 있습니다 — [modelcontextprotocol/servers](https://github.com/modelcontextprotocol/servers). 파일 시스템·깃허브·데이터베이스 같은 도구들이 이미 만들어져 있어, 필요한 것을 골라 위와 같은 방식으로 붙이면 됩니다. 규격 자체가 궁금하면 [공식 사양](https://modelcontextprotocol.io/docs/getting-started/intro) 을 보세요.

### 🖐️ 함께 따라하기 — 서버의 다른 도구 써 보기

데모에서는 환율 도구가 불렸습니다. 이번에는 **같은 서버의 다른 도구**를 쓰게 해 보세요.

1. `mcp_agent` 에 `'통계와 관련된 신간 도서를 찾아줘'` 를 물어봅니다(`asyncio.run(mcp_agent.ainvoke({'messages': ...}))`).
2. 불린 도구 이름을 출력합니다.
3. 최종 답(`.text`)을 출력합니다.

**확인 기준**: `search_new_books` 가 불리고, 최종 답에 책 제목이 들어간다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) asyncio.run 으로 mcp_agent.ainvoke 에 '통계와 관련된 신간 도서를 찾아줘' 를 넣는다
# 2) 불린 도구 이름을 출력한다
# 3) 최종 답(.text)을 출력한다

### ✅ 바로 확인 퀴즈

**1.** MCP 를 한 문장으로 설명하면?

<details><summary>정답 보기</summary>

**도구를 노출하고 부르는 표준 규격**입니다. 누가 만든 도구든 같은 방식으로 목록을 받아 같은 방식으로 부를 수 있게 해 줍니다(USB-C 처럼).

</details>

**2.** MCP 서버의 도구를 에이전트에 붙이는 큰 흐름은?

<details><summary>정답 보기</summary>

클라이언트로 서버에 연결해 **도구 목록을 받아 온 뒤**(`get_tools()`), 그 리스트를 `create_agent(model, mcp_tools)` 에 그대로 넘깁니다. 비동기라 실행은 `ainvoke` 를 `asyncio.run` 으로 감쌉니다.

</details>

**3.** 우리가 만든 `@tool` 과 MCP 서버가 준 도구를 함께 쓸 수 있나요?

<details><summary>정답 보기</summary>

있습니다. `mcp_tools + [내도구]` 처럼 **리스트를 합쳐** 넘기면 됩니다. 모델은 도구가 어디서 왔는지 구분하지 않고 이름·설명·인자 명세만 보고 고릅니다.

</details>

---
## 이번 강의 정리

| 개념 | 핵심 | 코드 |
|---|---|---|
| 구조화된 출력 | 답을 **정해진 칸**에 담아 받는다 | `model.with_structured_output(Ticket).invoke(문의)` |
| 스키마 설계(복습) | 값은 `Literal`, 없을 수 있으면 `Optional`, 필드마다 `description` | `Field(description=...)` |
| 표로 만들기(복습) | 객체 → 딕셔너리 → DataFrame | `pd.DataFrame([t.model_dump() for t in ...])` |
| 부품으로 재사용 | 분류기를 **한 번** 만들어 여러 문의에 돌린다. 모델 회사를 바꿔도 스키마·호출은 그대로 | `classifier = model.with_structured_output(Ticket)` |
| 도구 정의 | 함수 + `@tool`. docstring 이 명세, 타입힌트가 인자 타입 | `.name`·`.description`·`.args` |
| 도구 명세 | 모델에게는 JSON 으로 전달된다 | `.tool_call_schema.model_json_schema()` |
| 에이전트 | 도구 사용 여부·순서를 모델이 판단 | `create_agent(model, [도구들])` |
| 실행·궤적 | 질문 문자열 하나를 넣고 메시지를 읽는다 | `agent.invoke({'messages': '질문'})`, `ToolMessage`, `[-1].text` |
| 에이전트의 구조화 응답 | **도구를 쓴 뒤** 답까지 스키마로 | `response_format=ProviderStrategy(S, strict=True)` → `res['structured_response']` |
| 좋은 도구 | 인자 다듬기·실패는 문자열·좁은 권한 | 도구를 만들 때 지킬 것 |
| MCP | 도구의 **표준 규격** — 남의 도구를 그대로 붙인다 | `get_tools()` → `create_agent`, `asyncio.run(... .ainvoke(...))` |

- **판별 질문**: "이 답을 만들려면 우리 데이터를 뒤져야 하나?" — 그렇다면 도구, 아니라면 구조화된 출력.
- 도구를 만들 때 가장 오래 붙잡고 있어야 할 줄은 **docstring** 입니다 — 모델은 코드를 못 보고 이 설명만 봅니다.

## ⏭️ 예고 — 다음 시간

다음 시간에는 우리가 **이미 배운 것들을 도구로** 만듭니다 — **RAG 를 만든 단원**의 **검색**을 도구로, **SQL·데이터베이스 단원**의 **조회**를 도구로 붙여, 사내 문서와 사내 데이터를 함께 다루는 에이전트를 만듭니다. 만든 검색이 얼마나 맞히는지 **재 보는** 것까지 다음 시간에 함께 합니다.

수고하셨습니다!